# RL notebook (Kaggle T4) -- A7

`scripts/chat_rl.py` (GRPO-lite, GSM8K reward) trained on the *full* 7473-example GSM8K train
set with default settings would mean roughly `7473 examples x 16 samples/example` generations
just for rollouts, repeated across ~467 optimizer steps -- many hours on a single T4, for a model
whose GSM8K accuracy is already measured at ~0% (see A6 in the project plan). RL sharpens
existing capability; there's essentially no capability here to sharpen, so a full run is a bad
trade of GPU-hours for information.

Added `--max-train-examples` to `chat_rl.py` (wires up `tasks.common.Task`'s existing `stop`
kwarg, which wasn't exposed via CLI upstream) to run a bounded, honest-but-cheap version instead:
480 GSM8K examples, 8 samples each, 128 max generated tokens -- enough to get a real read on
whether reward/pass@k move at all, without gambling hours on a near-certain null result.

Targets `d6` only (our better-performing model from A3-A6) -- not `d4` too, to keep this within
budget; can be repeated for `d4` later if there's time.

Upload via File -> Upload Notebook. Same 4 Kaggle Secrets, T4 x2 accelerator (single GPU used,
same as kaggle_eval.ipynb -- chat_rl.py *can* run under torchrun for multi-GPU but isn't worth
the setup complexity at this scale), internet access.

## Cell 1: clone repo, install dependencies

In [4]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"
MODEL_TAG = "d6"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

Repo already present, pulling latest...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 860 bytes | 860.00 KiB/s, done.
From https://github.com/nadeko0/nanochat-ru
   abc3505..80aac7a  master     -> origin/master
Updating abc3505..80aac7a
Fast-forward
 kaggle/kaggle_rl.ipynb | 32 ++------------------------------
 1 file changed, 2 insertions(+), 30 deletions(-)
Using Python 3.12.13 environment at: /usr
Resolved 84 packages in 69ms                                         
Checked 84 packages in 2ms
Cell 1 done.


## Cell 2: configure rclone, pull the d6 SFT checkpoint + tokenizer

In [5]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

!rclone lsd gdrive:

NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

!rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v
!rclone copy gdrive:chatsft_checkpoints/{MODEL_TAG} {NANOCHAT_BASE_DIR}/chatsft_checkpoints/{MODEL_TAG} --checksum -v

print("Checkpoint ready:")
!ls {NANOCHAT_BASE_DIR}/chatsft_checkpoints/{MODEL_TAG}

           0 2026-08-10 16:16:45        -1 base_checkpoints
           0 2026-08-10 15:53:07        -1 base_data_climbmix
           0 2026-08-10 17:58:32        -1 chatsft_checkpoints
           0 2026-08-10 15:56:35        -1 tokenizer
2026/08/11 15:10:57 INFO  : There was nothing to transfer
2026/08/11 15:10:57 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 2 / 2, 100%, Listed 4
Elapsed time:         0.2s

2026/08/11 15:10:58 INFO  : There was nothing to transfer
2026/08/11 15:10:58 INFO  : 
Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 4 / 4, 100%, Listed 8
Elapsed time:         0.8s

Checkpoint ready:
meta_000063.json  model_000063.pt  optim_000063_rank0.pt  optim_000063_rank1.pt


## Cell 3: RL (bounded), with a background watcher syncing new checkpoints to Drive

In [6]:
import os
import subprocess
import sys

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)

SYNC_LOG = "/kaggle/working/sync_checkpoints.log"
sync_proc = subprocess.Popen(
    [sys.executable, "kaggle/sync_checkpoints.py", "--remote", "gdrive:", "--interval", "120", "--log-file", SYNC_LOG],
    env=os.environ.copy(),
)
print(f"Started background checkpoint sync watcher, pid={sync_proc.pid}")

# No "--" separator here (unlike the torchrun-launched training scripts) -- "--" is torchrun's
# own convention for splitting its args from the script's; plain `python -m` doesn't use it and
# argparse chokes on a literal "--" token.
rl_cmd = (
    "python -m scripts.chat_rl "
    f"--model-tag={MODEL_TAG} --max-train-examples=480 --num-samples=8 "
    "--device-batch-size=8 --max-new-tokens=128 --eval-examples=100 --run=dummy"
)
print(f"Running: {rl_cmd}")
try:
    !{rl_cmd}
finally:
    sync_proc.terminate()
    sync_proc.wait()
    !python kaggle/sync_checkpoints.py --remote gdrive: --once --log-file {SYNC_LOG}
    print("RL cell finished (or was interrupted), sync watcher stopped.")

Started background checkpoint sync watcher, pid=453
Running: python -m scripts.chat_rl --model-tag=d6 --max-train-examples=480 --num-samples=8 --device-batch-size=8 --max-new-tokens=128 --eval-examples=100 --run=dummy
sync_checkpoints: base_dir=/kaggle/working/nanochat_cache remote=gdrive: interval=120s once=False
[2026-08-11 15:10:59] OK   tokenizer -> gdrive:/tokenizer
[2026-08-11 15:11:01] OK   chatsft_checkpoints -> gdrive:/chatsft_checkpoints
Autodetected device type: cuda
2026-08-11 15:11:04,517 - nanochat.common - INFO - Distributed world size: 1
2026-08-11 15:11:04,518 - nanochat.checkpoint_manager - INFO - Loading model from /kaggle/working/nanochat_cache/chatsft_checkpoints/d6 with step 63
2026-08-11 15:11:05,025 - nanochat.checkpoint_manager - INFO - Building model with config: {'sequence_len': 2048, 'vocab_size': 32768, 'n_layer': 6, 'n_head': 3, 'n_kv_head': 3, 'n_embd': 384, 'window_pattern': 'L'}
2026-08-11 15:11:06,692 - numexpr.utils - INFO - NumExpr defaulting to 4 th